# Zadanie 1: Klasyfikacja binarna - Diagnoza raka piersi

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

import optuna
import tqdm

c:\VScode\Python\StatisticalLearning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("../Data/wisconsin_breast_cancer.csv")
data.head()

,id,thickness,size,shape,adhesion,single,nuclei,chromatin,nucleoli,mitosis,class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0


In [3]:
data.shape

(699, 11)

In [4]:
data.isna().sum()



id            0
thickness     0
size          0
shape         0
adhesion      0
single        0
nuclei       16
chromatin     0
nucleoli      0
mitosis       0
class         0
dtype: int64

In [5]:
data['nuclei'] = pd.to_numeric(data['nuclei'], errors='coerce')

In [6]:
display(data[data['nuclei'].isna()])

,id,thickness,size,shape,adhesion,single,nuclei,chromatin,nucleoli,mitosis,class
23,1057013,8,4,5,1,2,NaN,7,3,1,1
40,1096800,6,6,6,9,6,NaN,7,8,1,0
139,1183246,1,1,1,1,1,NaN,2,1,1,0
145,1184840,1,1,3,1,2,NaN,2,1,1,0
158,1193683,1,1,2,1,3,NaN,1,1,1,0
164,1197510,5,1,1,1,2,NaN,3,1,1,0
235,1241232,3,1,4,1,2,NaN,3,1,1,0
249,169356,3,1,1,1,2,NaN,3,1,1,0
275,432809,3,1,3,1,2,NaN,2,1,1,0
292,563649,8,8,8,1,2,NaN,6,10,1,1


In [7]:
data['nuclei'].fillna(data['nuclei'].median(), inplace=True)
data.isna().sum()

C:\Users\posze\AppData\Local\Temp\ipykernel_5384\2179343237.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['nuclei'].fillna(data['nuclei'].median(), inplace=True)


id           0
thickness    0
size         0
shape        0
adhesion     0
single       0
nuclei       0
chromatin    0
nucleoli     0
mitosis      0
class        0
dtype: int64

In [8]:
X = data.drop(['id','class'],axis=1)
y = data['class']

In [9]:
X_train1,X_val,y_train1,y_val = train_test_split(X,y,test_size=.2,shuffle=False,random_state=42)

In [10]:
def best_model_creator(X_train,y_train,model_name):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial:optuna.trial.Trial):
        n_features = trial.suggest_int("n_features",3,X_train.shape[1])

        if model_name == "LR":
            penalty = trial.suggest_categorical('penalty',[None,'l2','l1'])

            if penalty == 'l1':
                solver = 'liblinear'
            else:
                solver = 'lbfgs'

            model = LogisticRegression(penalty=penalty,solver=solver)
        elif model_name == "KNN":
            n_neighbors = trial.suggest_int("n_neighbors",3,5)
            weights = trial.suggest_categorical("weights",['uniform','distance'])
            model = KNeighborsClassifier(n_neighbors=n_neighbors,weights=weights)

        pipeline = Pipeline([
            ("StandardScaler",StandardScaler()),
            ("SelectKBest",SelectKBest(k=n_features,score_func=f_classif)),
            ("Classifier",model)
        ])

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=skf,
            scoring="roc_auc"
        )

        return scores.mean()
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective,n_trials=30)

    return study.best_params,study.best_value

In [11]:
val_cv = StratifiedKFold(n_splits=5,shuffle=True)

In [12]:
results = {
    "accuracy_lr": [],
    "accuracy_knn": [],
    "precission_lr": [],
    "precission_knn": [],
    "recall_lr": [],
    "recall_knn": [],
    "f1_lr": [],
    "f1_knn": [],
}

In [13]:
fold_idx = 0

for train_idx,test_idx in val_cv.split(X_train1,y_train1):
    fold_idx += 1
    X_train,X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train,y_test = y.iloc[train_idx], y.iloc[test_idx]

    best_params_lr,best_score_lr = best_model_creator(X_train,y_train,"LR")

    pipeline = Pipeline([
        ("StandardScaler",StandardScaler()),
        ("SelectKBest",SelectKBest(k=best_params_lr["n_features"],score_func=f_classif)),
        ("classificator", LogisticRegression(
            penalty=best_params_lr["penalty"],
            solver='liblinear' if best_params_lr["penalty"]=='l1' else 'lbfgs'
        ))
    ])

    pipeline.fit(X_train,y_train)
    preds_lr = pipeline.predict(X_test)

    TP = np.sum((preds_lr == 1) & (y_test == 1))
    TN = np.sum((preds_lr == 0) & (y_test == 0))
    FP = np.sum((preds_lr == 1) & (y_test == 0))
    FN = np.sum((preds_lr == 0) & (y_test == 1))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precission = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = f1_score(y_test,preds_lr)
    
    results["accuracy_lr"].append(accuracy)
    results["precission_lr"].append(precission)
    results["recall_lr"].append(recall)
    results["f1_lr"].append(f1)


    best_params_knn,best_score_knn = best_model_creator(X_train,y_train,"KNN")
    pipeline = Pipeline([
        ("StandardScaler",StandardScaler()),
        ("SelectKBest",SelectKBest(k=best_params_knn["n_features"],score_func=f_classif)),
        ("classificator", KNeighborsClassifier(
            n_neighbors=best_params_knn["n_neighbors"],
            weights=best_params_knn["weights"]
        ))
    ])
    pipeline.fit(X_train,y_train)
    preds_knn = pipeline.predict(X_test)

    TP = np.sum((preds_knn == 1) & (y_test == 1))
    TN = np.sum((preds_knn == 0) & (y_test == 0))
    FP = np.sum((preds_knn == 1) & (y_test == 0))
    FN = np.sum((preds_knn == 0) & (y_test == 1))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precission = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = f1_score(y_test,preds_knn)

    results["accuracy_knn"].append(accuracy)
    results["precission_knn"].append(precission)
    results["recall_knn"].append(recall)
    results["f1_knn"].append(f1)
    print(f"Fold {fold_idx} completed.")

for key in results.keys():
    results[key] = np.mean(results[key])

results    

[I 2025-11-25 09:03:39,011] A new study created in memory with name: no-name-978eb864-655c-4974-b7ea-2fc9a3ab68a8
[I 2025-11-25 09:03:39,061] Trial 0 finished with value: 0.9902274625958837 and parameters: {'n_features': 3, 'penalty': 'l1'}. Best is trial 0 with value: 0.9902274625958837.
[I 2025-11-25 09:03:39,117] Trial 1 finished with value: 0.9938786359838993 and parameters: {'n_features': 5, 'penalty': 'l1'}. Best is trial 1 with value: 0.9938786359838993.
[I 2025-11-25 09:03:39,242] Trial 2 finished with value: 0.9929084073820915 and parameters: {'n_features': 7, 'penalty': 'l2'}. Best is trial 1 with value: 0.9938786359838993.
[I 2025-11-25 09:03:39,372] Trial 3 finished with value: 0.9936507936507937 and parameters: {'n_features': 9, 'penalty': 'l2'}. Best is trial 1 with value: 0.9938786359838993.
[I 2025-11-25 09:03:39,455] Trial 4 finished with value: 0.9935577580314423 and parameters: {'n_features': 4, 'penalty': 'l1'}. Best is trial 1 with value: 0.9938786359838993.
[I 202

Fold 1 completed.


[I 2025-11-25 09:03:44,315] Trial 3 finished with value: 0.9937476266423635 and parameters: {'n_features': 8, 'penalty': 'l2'}. Best is trial 2 with value: 0.9939564821143769.
[I 2025-11-25 09:03:44,360] Trial 4 finished with value: 0.9898505734032049 and parameters: {'n_features': 3, 'penalty': 'l1'}. Best is trial 2 with value: 0.9939564821143769.
[I 2025-11-25 09:03:44,430] Trial 5 finished with value: 0.9941634389002811 and parameters: {'n_features': 6, 'penalty': 'l2'}. Best is trial 5 with value: 0.9941634389002811.
[I 2025-11-25 09:03:44,476] Trial 6 finished with value: 0.9938482570061519 and parameters: {'n_features': 9, 'penalty': 'l1'}. Best is trial 5 with value: 0.9941634389002811.
[I 2025-11-25 09:03:44,551] Trial 7 finished with value: 0.9927565124933546 and parameters: {'n_features': 5, 'penalty': None}. Best is trial 5 with value: 0.9941634389002811.
[I 2025-11-25 09:03:44,632] Trial 8 finished with value: 0.9926596795017847 and parameters: {'n_features': 9, 'penalty':

Fold 2 completed.


[I 2025-11-25 09:03:49,570] Trial 2 finished with value: 0.9904458114984431 and parameters: {'n_features': 8, 'penalty': None}. Best is trial 2 with value: 0.9904458114984431.
[I 2025-11-25 09:03:49,733] Trial 3 finished with value: 0.9895762132604237 and parameters: {'n_features': 4, 'penalty': None}. Best is trial 2 with value: 0.9904458114984431.
[I 2025-11-25 09:03:49,802] Trial 4 finished with value: 0.9913097136781346 and parameters: {'n_features': 7, 'penalty': 'l1'}. Best is trial 4 with value: 0.9913097136781346.
[I 2025-11-25 09:03:49,901] Trial 5 finished with value: 0.990776182881446 and parameters: {'n_features': 7, 'penalty': None}. Best is trial 4 with value: 0.9913097136781346.
[I 2025-11-25 09:03:49,991] Trial 6 finished with value: 0.9915299612668033 and parameters: {'n_features': 9, 'penalty': 'l2'}. Best is trial 6 with value: 0.9915299612668033.
[I 2025-11-25 09:03:50,080] Trial 7 finished with value: 0.9881237183868763 and parameters: {'n_features': 3, 'penalty': 

...

[I 2025-11-25 09:03:54,622] Trial 2 finished with value: 0.9894105528973951 and parameters: {'n_features': 9, 'penalty': None}. Best is trial 1 with value: 0.9920150755677073.
[I 2025-11-25 09:03:54,728] Trial 3 finished with value: 0.9907206463127516 and parameters: {'n_features': 8, 'penalty': None}. Best is trial 1 with value: 0.9920150755677073.
[I 2025-11-25 09:03:54,810] Trial 4 finished with value: 0.9896464646464647 and parameters: {'n_features': 4, 'penalty': 'l2'}. Best is trial 1 with value: 0.9920150755677073.
[I 2025-11-25 09:03:54,894] Trial 5 finished with value: 0.9920278916989445 and parameters: {'n_features': 6, 'penalty': 'l2'}. Best is trial 5 with value: 0.9920278916989445.
[I 2025-11-25 09:03:54,968] Trial 6 finished with value: 0.9916956216298323 and parameters: {'n_features': 8, 'penalty': 'l1'}. Best is trial 5 with value: 0.9920278916989445.
[I 2025-11-25 09:03:55,029] Trial 7 finished with value: 0.9915873965216072 and parameters: {'n_features': 8, 'penalty':

Fold 4 completed.


[I 2025-11-25 09:03:59,506] Trial 2 finished with value: 0.9933337130705551 and parameters: {'n_features': 8, 'penalty': 'l1'}. Best is trial 0 with value: 0.9933337130705551.
[I 2025-11-25 09:03:59,603] Trial 3 finished with value: 0.9939792663476872 and parameters: {'n_features': 9, 'penalty': None}. Best is trial 3 with value: 0.9939792663476872.
[I 2025-11-25 09:03:59,664] Trial 4 finished with value: 0.9918337510442775 and parameters: {'n_features': 5, 'penalty': 'l1'}. Best is trial 3 with value: 0.9939792663476872.
[I 2025-11-25 09:03:59,749] Trial 5 finished with value: 0.9933337130705551 and parameters: {'n_features': 8, 'penalty': 'l2'}. Best is trial 3 with value: 0.9939792663476872.
[I 2025-11-25 09:03:59,845] Trial 6 finished with value: 0.9922666514771776 and parameters: {'n_features': 5, 'penalty': 'l2'}. Best is trial 3 with value: 0.9939792663476872.
[I 2025-11-25 09:03:59,913] Trial 7 finished with value: 0.9925704412546518 and parameters: {'n_features': 6, 'penalty':

Fold 5 completed.


{'accuracy_lr': np.float64(0.9535070785070786),
 'accuracy_knn': np.float64(0.9606177606177605),
 'precission_lr': np.float64(0.9416003352112556),
 'precission_knn': np.float64(0.9384220619614834),
 'recall_lr': np.float64(0.9320557491289199),
 'recall_knn': np.float64(0.9564459930313589),
 'f1_lr': np.float64(0.9364274034542636),
 'f1_knn': np.float64(0.9472734142143487)}

In [14]:
import scipy.stats as stats

In [15]:
def mcnemar_test(y_true, y_pred1, y_pred2):
    correct1 = (y_pred1 == y_true)
    correct2 = (y_pred2 == y_true)

    n00 = np.sum((correct1 == False) & (correct2 == False))
    n01 = np.sum((correct1 == False) & (correct2 == True))
    n10 = np.sum((correct1 == True) & (correct2 == False))
    n11 = np.sum((correct1 == True) & (correct2 == True))

    contingency_table = np.array([[n00, n01],
                                  [n10, n11]])
    
    statistic = (abs(n01- n10) - 1)**2 / (n01 + n10) if (n01 + n10) != 0 else 0.0

    p_value = 1 - stats.chi2.cdf(statistic, df=1)

    return p_value, contingency_table

In [16]:
preds_lr_final = pipeline.predict(X_val)
preds_knn_final = pipeline.predict(X_val)

p_value, contingency_table = mcnemar_test(y_val, preds_lr_final, preds_knn_final)

print("Contingency Table:")
print(contingency_table)
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("Różnica jest statystycznie istotna.")
else:
    print("Różnica nie jest statystycznie istotna.")

Contingency Table:
[[  2   0]
 [  0 138]]
P-value: 1.0
Różnica nie jest statystycznie istotna.
